In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import scipy.stats as stats

# --- 1. Load and Prep Data ---
coords = pd.read_csv('..//data/COORDINATE_XY.csv')
classification = pd.read_csv('..//data/classification.csv')
df = coords.merge(classification[['root_id', 'super_class']], on='root_id', how='inner')
df.rename(columns={'y_coord': 'score', 'super_class': 'target'}, inplace=True)

# Encode target classes
le = LabelEncoder()
df['target_encoded'] = le.fit_transform(df['target'])

# Train Decision Tree
tree = DecisionTreeClassifier(criterion='entropy', max_leaf_nodes=5, min_samples_leaf=50, random_state=42)
tree.fit(df[['score']], df['target_encoded'])

# Assign logical hierarchical levels based on scores
df['level'] = tree.apply(df[['score']])
level_order = df.groupby('level')['score'].mean().sort_values(ascending=False).index
mapping = {code: i + 1 for i, code in enumerate(level_order)}
df['y_level'] = df['level'].map(mapping)

print("======================================================")
print("1. DECISION TREE CLASSIFIER METRICS")
print("======================================================")
y_pred = tree.predict(df[['score']])
y_true = df['target_encoded']

accuracy = accuracy_score(y_true, y_pred)
print(f"Overall Accuracy of hierarchical splits: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Detailed Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))


print("\n======================================================")
print("2. STATISTICAL SIGNIFICANCE (KRUSKAL-WALLIS TEST)")
print("======================================================")
# Group the continuous scores by superclass
groups = [group['score'].values for name, group in df.groupby('target')]

# Perform Kruskal-Wallis H-test (non-parametric ANOVA for distributions)
H_stat, p_value = stats.kruskal(*groups)
print(f"Kruskal-Wallis H-statistic: {H_stat:.2f}")
print(f"p-value: {p_value:.2e}")

if p_value < 0.05:
    print("Conclusion: The separation of superclasses across the hierarchical score is STATISTICALLY SIGNIFICANT.")
else:
    print("Conclusion: No significant difference found across superclasses.")


print("\n======================================================")
print("3. HIERARCHICAL EDGE DIRECTION (FLOW VALIDATION)")
print("======================================================")
try:
    df_conn = pd.read_csv('..//data/connections.csv')
    
    # Map root_id to hierarchical level (y_level: 1 to 5)
    level_dict = df.set_index('root_id')['y_level'].to_dict()

    df_conn['pre_level'] = df_conn['pre_root_id'].map(level_dict)
    df_conn['post_level'] = df_conn['post_root_id'].map(level_dict)

    # Drop edges where nodes don't have a level assigned
    valid_edges = df_conn.dropna(subset=['pre_level', 'post_level'])
    total_synapses = valid_edges['syn_count'].sum()

    # Feedforward: goes from lower level index (e.g. L1) to higher (e.g. L3)
    ff_synapses = valid_edges[valid_edges['pre_level'] < valid_edges['post_level']]['syn_count'].sum()

    # Feedback: goes from higher level (e.g. L4) to lower (e.g. L2)
    fb_synapses = valid_edges[valid_edges['pre_level'] > valid_edges['post_level']]['syn_count'].sum()

    # Lateral: stays within the same level (e.g. L2 -> L2)
    lat_synapses = valid_edges[valid_edges['pre_level'] == valid_edges['post_level']]['syn_count'].sum()

    print(f"Total Synapses Analyzed: {total_synapses:,}")
    print(f"Feedforward (FF) Synapses : {ff_synapses:,} ({ff_synapses/total_synapses*100:.2f}%)")
    print(f"Feedback (FB) Synapses    : {fb_synapses:,} ({fb_synapses/total_synapses*100:.2f}%)")
    print(f"Lateral (LAT) Synapses    : {lat_synapses:,} ({lat_synapses/total_synapses*100:.2f}%)")

    if fb_synapses > 0:
        print(f"\nFF to FB Ratio: {ff_synapses/fb_synapses:.2f} (For every 1 feedback synapse, there are {ff_synapses/fb_synapses:.2f} feedforward synapses)")
        
except FileNotFoundError:
    print("Error: Could not find '..//data/connections.csv'.")

1. DECISION TREE CLASSIFIER METRICS
Overall Accuracy of hierarchical splits: 0.6665 (66.65%)

Classification Report:
                    precision    recall  f1-score   support

         ascending       0.00      0.00      0.00      1989
           central       0.58      0.52      0.55     32298
        descending       0.00      0.00      0.00      1276
         endocrine       0.00      0.00      0.00        69
             motor       0.00      0.00      0.00       106
             optic       0.69      0.94      0.80     77558
           sensory       0.00      0.00      0.00     12711
visual_centrifugal       0.00      0.00      0.00       516
 visual_projection       0.00      0.00      0.00      7658

          accuracy                           0.67    134181
         macro avg       0.14      0.16      0.15    134181
      weighted avg       0.54      0.67      0.59    134181


2. STATISTICAL SIGNIFICANCE (KRUSKAL-WALLIS TEST)
Kruskal-Wallis H-statistic: 37450.88
p-value: 0.0